In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from collections import defaultdict
import pytz

dam_meta = pd.read_csv('your_directory/hourly_flow_metadata.csv') 
dam_meta['geometry'] = [Point(xy) for xy in zip(dam_meta['longitude'], dam_meta['latitude'])]
dam_gdf = gpd.GeoDataFrame(dam_meta, geometry='geometry', crs='EPSG:4326')

county_gdf = gpd.read_file('your_directory/tl_2020_us_county.shp').to_crs('EPSG:4326')
county_gdf = county_gdf[['GEOID', 'geometry']]
joined_gdf = gpd.sjoin(dam_gdf, county_gdf, how='left', predicate='within')
dam_to_county = dict(zip(joined_gdf['dam_name'], joined_gdf['GEOID']))

tz_gdf = gpd.read_file('your_directory/tz_us.shp')[['TZID', 'geometry']].to_crs('EPSG:4326')
dam_with_tz = gpd.sjoin(dam_gdf, tz_gdf, how='left', predicate='within')
dam_to_tz = dict(zip(dam_with_tz['dam_name'], dam_with_tz['TZID']))

df = pd.read_csv('your_directory/hourly_flow.csv')  
df.columns = ['local_time', 'dam_name', 'power', 'discharge']

In [ ]:
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

def convert_to_utc(args):
    local_time_str, dam, dam_to_tz = args
    tzname = dam_to_tz.get(dam, 'America/New_York') 
    try:
        naive_local_time = pd.to_datetime(local_time_str).replace(tzinfo=None)
        local_dt = pytz.timezone(tzname).localize(naive_local_time, is_dst=None)
        utc_dt = local_dt.astimezone(pytz.UTC)
        return utc_dt
    except Exception as e:
        return pd.NaT 

df = pd.read_csv('your_directory/hourly_flow.csv')
df.columns = ['local_time', 'dam_name', 'power', 'discharge']
args_list = list(zip(df['local_time'], df['dam_name'], [dam_to_tz]*len(df)))

print("local_time → UTC...")
with Pool(cpu_count()) as pool:
    utc_times = list(tqdm(pool.imap(convert_to_utc, args_list, chunksize=1000),
                          total=len(df), desc="Converting to UTC (parallel)"))
df['utc_time'] = pd.Series(utc_times)
df = df.dropna(subset=['utc_time'])

In [5]:
df['year'] = df['utc_time'].dt.year
df['hour_of_year'] = df['utc_time'].dt.dayofyear * 24 + df['utc_time'].dt.hour - 24  # 0-based hour

dam_curves = {}
for dam, group in df.groupby('dam_name'):
    group = group[(group['hour_of_year'] >= 0) & (group['hour_of_year'] < 8760)]
    pivot = group.pivot_table(index='hour_of_year', columns='year', values='power')
    all_hours = pd.Series(range(8760), name='hour_of_year')
    pivot = pivot.reindex(all_hours)
    avg_curve = pivot.mean(axis=1)
    dam_curves[dam] = avg_curve.ffill().bfill().values

county_power = defaultdict(list)
for dam, curve in dam_curves.items():
    county = dam_to_county.get(dam)
    if county is not None:
        county_power[county].append(curve)
county_curves = {
    county: np.stack(curves).mean(axis=0)
    for county, curves in county_power.items()
}

for dam, arr in dam_curves.items():
    if len(arr) != 8760:
        print(f"[Dam] {dam} has length {len(arr)}")
for county, arr in county_curves.items():
    if len(arr) != 8760:
        print(f"[County] {county} has length {len(arr)}")
np.savez_compressed('your_directory/average_dam_curves.npz', **dam_curves)
np.savez_compressed('your_directory/average_county_curves.npz', **county_curves)

In [ ]:
df = pd.read_csv('your_directory/hourly_flow.csv') 
df.columns = ['local_time', 'dam_name', 'power', 'discharge']

df['utc_time'] = pd.to_datetime(df['local_time'], utc=True)
df['year'] = df['utc_time'].dt.year
df['hour'] = df['utc_time']

results = {}
for dam, group in df.groupby('dam_name'):
    group_sorted = group.sort_values('utc_time')
    start = group_sorted['utc_time'].iloc[0]
    end = group_sorted['utc_time'].iloc[-1]
    total_hours = group_sorted.shape[0]
    hours_by_year = group_sorted.groupby(group_sorted['utc_time'].dt.year).size().to_dict()
    full_years = {y: h for y, h in hours_by_year.items() if h in [8760, 8784]}
    hourly_diff = group_sorted['utc_time'].diff().dropna()
    gaps = hourly_diff[hourly_diff != pd.Timedelta(hours=1)].count()
    results[dam] = {
        'start': str(start),
        'end': str(end),
        'total_hours': total_hours,
        'yearly_counts': hours_by_year,
        'full_years': list(full_years.keys()),
        'missing_hour_gaps': int(gaps)
    }


print("=== Dams with incomplete data (less than 8760 hours total or with time gaps): ===")
for dam, info in results.items():
    if info['total_hours'] < 8760 or info['missing_hour_gaps'] > 0:
        print(f"[{dam}] {info['total_hours']} hours from {info['start']} to {info['end']} | gaps: {info['missing_hour_gaps']} | years: {info['yearly_counts']}")

In [ ]:
import matplotlib.pyplot as plt
import random

county_curves = np.load('your_directory/average_county_curves.npz')

all_curves = np.stack([county_curves[key] for key in county_curves if len(county_curves[key]) == 8760])
national_curve = all_curves.sum(axis=0)

utc_index = pd.date_range(start='2011-01-01', periods=8760, freq='H', tz='UTC')
est_index = utc_index.tz_convert('America/New_York')

plt.figure(figsize=(15, 4))
plt.plot(national_curve*1.5, lw=0.5)
plt.title("Nationwide Total Hourly Power Generation (Average over 2011–2016)")
plt.xlabel("Time (Eastern Time)")
plt.ylabel("Power (MW or Unit)")
plt.grid(True)
plt.tight_layout()
plt.show()

start_hour = random.randint(0, 8760 - 168)
end_hour = start_hour + 168

week_index = est_index[start_hour:end_hour]
week_curve = national_curve[start_hour:end_hour]

plt.figure(figsize=(12, 4))
plt.plot(week_curve)
plt.title(f"Random Week of Power Generation ({week_index[0].strftime('%Y-%m-%d')} to {week_index[-1].strftime('%Y-%m-%d')})")
plt.xlabel("Time (Eastern Time)")
plt.ylabel("Power")
plt.grid(True)
plt.tight_layout()
plt.show()